# **Tahap 1 - Ekstrak data mentah menjadi CSV**

## Import Library

In [16]:
import re
try:
    import pdfplumber
except ModuleNotFoundError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pdfplumber"])
    import pdfplumber
import pandas as pd
from pathlib import Path

## Path Konfigurasi

In [17]:
BASE_DIR      = Path(__file__).resolve().parent.parent \
                if "__file__" in dir() else Path.cwd().parent
 
RAW_DIR       = BASE_DIR / "data" / "raw"
EXTRACTED_DIR = BASE_DIR / "data" / "extracted"
 
# Validasi — pastikan folder raw ada sebelum lanjut
if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"Folder data/raw tidak ditemukan: {RAW_DIR}\n"
        f"Pastikan kamu menjalankan script dari dalam folder project:\n"
        f"  cd crime-indonesia-analysis\n"
        f"  python src/01_extract_pdf.py"
    )
 
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

## Ekstrak Crime Data from PDF

### Mapping nama Polda (di PDF) → nama Provinsi resmi

In [18]:
POLDA_TO_PROVINSI = {
    "A c e h"                  : "Aceh",
    "Metro Jaya"               : "DKI Jakarta",
    "Kep. Bangka Belitung"     : "Kepulauan Bangka Belitung",
    "Kep Bangka Belitung"      : "Kepulauan Bangka Belitung",

    "Aceh"                     : "Aceh",
    "Sumatera Utara"           : "Sumatera Utara",
    "Sumatera Barat"           : "Sumatera Barat",
    "Riau"                     : "Riau",
    "Jambi"                    : "Jambi",
    "Sumatera Selatan"         : "Sumatera Selatan",
    "Bengkulu"                 : "Bengkulu",
    "Lampung"                  : "Lampung",
    "Kepulauan Bangka Belitung": "Kepulauan Bangka Belitung",
    "Kepulauan Riau"           : "Kepulauan Riau",
    "Jawa Barat"               : "Jawa Barat",
    "Jawa Tengah"              : "Jawa Tengah",
    "DI Yogyakarta"            : "DI Yogyakarta",
    "Jawa Timur"               : "Jawa Timur",
    "Banten"                   : "Banten",
    "Bali"                     : "Bali",
    "Nusa Tenggara Barat"      : "Nusa Tenggara Barat",
    "Nusa Tenggara Timur"      : "Nusa Tenggara Timur",
    "Kalimantan Barat"         : "Kalimantan Barat",
    "Kalimantan Tengah"        : "Kalimantan Tengah",
    "Kalimantan Selatan"       : "Kalimantan Selatan",
    "Kalimantan Timur"         : "Kalimantan Timur",
    "Kalimantan Utara"         : "Kalimantan Utara",
    "Sulawesi Utara"           : "Sulawesi Utara",
    "Sulawesi Tengah"          : "Sulawesi Tengah",
    "Sulawesi Selatan"         : "Sulawesi Selatan",
    "Sulawesi Tenggara"        : "Sulawesi Tenggara",
    "Gorontalo"                : "Gorontalo",
    "Sulawesi Barat"           : "Sulawesi Barat",
    "Maluku"                   : "Maluku",
    "Maluku Utara"             : "Maluku Utara",
    "Papua Barat Daya"         : "Papua Barat Daya",
    "Papua Barat"              : "Papua Barat",
    "Papua Selatan"            : "Papua Selatan",
    "Papua Tengah"             : "Papua Tengah",
    "Papua Pegunungan"         : "Papua Pegunungan",
    "Papua"                    : "Papua",
}

SORTED_POLDA = sorted(POLDA_TO_PROVINSI.keys(), key=len, reverse=True)

### Source PDF

In [19]:
PDF_SOURCES = [
    ("BPS-statistik-kriminal-2018.pdf",     98,  [2015, 2016, 2017]),
    ("BPS-statistik-kriminal-2021.pdf",     118, [2018, 2019, 2020]),
    ("BPS-statistik-kriminal-2022.pdf",     110, [2019, 2020, 2021]),  
    ("BPS-statistik-kriminal-2023.pdf",     118, [2020, 2021, 2022]),
    ("BPS-statistik-kriminal-2024.pdf",     122, [2022, 2023]),
    ("BPS-statistik-kriminal-2024-2025.pdf",125, [2023, 2024]),
]
 

### Cleaning

In [20]:
def _clean_number_string(s: str) -> str:
    s = re.sub(r'(\d+)\.(\d+)\.(\d{2,3})',
               lambda m: m.group(1) + m.group(2) + m.group(3), s)
    s = re.sub(r'\d+\)', ' ', s)
    s = re.sub(r'[^\d.\s]', ' ', s)
    s = re.sub(r'\.(?!\d{3})', ' ', s)
    s = re.sub(r'(\d)\.(\d{3})(?!\d)', r'\1\2', s)
    return re.sub(r'\s+', ' ', s).strip()

### Parsing Numbers

In [21]:
def _parse_numbers(text: str, expected_count: int) -> list:
    s = _clean_number_string(text)
    tokens = [t for t in s.split() if re.match(r'^\d+$', t)]
    if not tokens:
        return []
 
    result = []
    i = 0
    while i < len(tokens):
        cur = tokens[i]
        nxt = tokens[i + 1] if i + 1 < len(tokens) else None
        if nxt and 1 <= len(cur) <= 3 and len(nxt) == 3:
            result.append(int(cur + nxt))
            i += 2
        else:
            result.append(int(cur))
            i += 1

    valid = [n for n in result if 0 < n < 500_000]
    return valid[:expected_count]

### Extracting Table from Page

In [22]:
def _extract_table_from_page(text: str, num_years: int) -> dict:
    rows = {}
    for raw_line in text.split('\n'):
        line = raw_line.strip()
        if not line:
            continue
        for polda_name in SORTED_POLDA:
            if line.startswith(polda_name):
                provinsi = POLDA_TO_PROVINSI[polda_name]
                remainder = line[len(polda_name):]
                remainder = re.sub(r'^\d\s', ' ', remainder)
                nums = _parse_numbers(remainder, num_years)
                if len(nums) == num_years:
                    rows[provinsi] = nums
                break
    return rows

### Extract Crime from PDF

In [23]:
def extract_crime_from_pdf(pdf_path: Path, page_idx: int, years: list) -> dict:
    result = {}
    try:
        with pdfplumber.open(pdf_path) as pdf:
            if page_idx >= len(pdf.pages):
                print(f"  [WARN] Halaman {page_idx+1} tidak ada di {pdf_path.name}")
                return result
            text = pdf.pages[page_idx].extract_text() or ''
            rows = _extract_table_from_page(text, len(years))
            for provinsi, vals in rows.items():
                for i, year in enumerate(years):
                    if i < len(vals):
                        result[(provinsi, year)] = vals[i]
    except Exception as e:
        print(f"  [ERROR] {pdf_path.name}: {e}")
    return result

### Build CSV for Crime

In [24]:
def build_crime_csv() -> pd.DataFrame:

    print("=" * 60)
    print("EKSTRAKSI CRIME DATA DARI PDF BPS")
    print("=" * 60)

    all_data = {}  

    for pdf_file, page_idx, years in PDF_SOURCES:
        pdf_path = RAW_DIR / pdf_file
        if not pdf_path.exists():
            print(f"  [SKIP] File tidak ditemukan: {pdf_file}")
            continue

        extracted = extract_crime_from_pdf(pdf_path, page_idx, years)
        all_data.update(extracted)  # last wins
        print(f"  ✓ {pdf_file[-35:]:<40} hal.{page_idx+1}  tahun{years}  → {len(extracted)//len(years)} provinsi")
        
    records = [
        {"provinsi": k[0], "tahun": k[1], "crime_total": v}
        for k, v in all_data.items()
    ]
    df = pd.DataFrame(records).sort_values(["provinsi", "tahun"]).reset_index(drop=True)

    print(f"\nHasil: {len(df)} records | {df['provinsi'].nunique()} provinsi | tahun {df['tahun'].min()}–{df['tahun'].max()}")
    return df

## Ekstrak Populasi dari BPS (CSV)

In [25]:
def build_population_csv() -> pd.DataFrame:
    print("\n" + "=" * 60)
    print("EKSTRAKSI DATA POPULASI DARI CSV BPS")
    print("=" * 60)
 
    frames = []
    pop_files = sorted(RAW_DIR.glob("Population*.csv"))
 
    for filepath in pop_files:
        year_match = re.search(r'(\d{4})\.csv$', filepath.name)
        if not year_match:
            continue
        year = int(year_match.group(1))
 
        try:
            df = pd.read_csv(filepath, skipfooter=10, engine='python')
            prov_col = next((c for c in df.columns if 'Province' in c or 'Provinsi' in c), None)
            pop_col = next((c for c in df.columns if 'Population' in c and 'Thousand' in c), None)
            density_col = next((c for c in df.columns if 'Density' in c), None)
 
            if not prov_col or not pop_col:
                print(f"  [WARN] Kolom tidak ditemukan di {filepath.name}")
                continue
 
            cols_to_use = [prov_col, pop_col]
            rename_map = {prov_col: 'provinsi', pop_col: 'population_thousands'}
 
            if density_col:
                cols_to_use.append(density_col)
                rename_map[density_col] = 'population_density'
 
            df = df[cols_to_use].rename(columns=rename_map)
            df = df.dropna(subset=['provinsi'])
            df = df[~df['provinsi'].isin(['Indonesia', 'INDONESIA'])]
            df['population_thousands'] = pd.to_numeric(df['population_thousands'], errors='coerce')
            df = df.dropna(subset=['population_thousands'])
            df['tahun'] = year
            frames.append(df)
            print(f"  ✓ {filepath.name:<65} → {len(df)} provinsi")
 
        except Exception as e:
            print(f"  [ERROR] {filepath.name}: {e}")
 
    df_pop = pd.concat(frames, ignore_index=True)

    rename_prov = {
        'D.K.I. Jakarta'           : 'DKI Jakarta',
        'D.I. Yogyakarta'          : 'DI Yogyakarta',
        'DI. Yogyakarta'           : 'DI Yogyakarta',
        'Yogyakarta'               : 'DI Yogyakarta',
        'Kep. Bangka Belitung'     : 'Kepulauan Bangka Belitung',
        'Bangka Belitung Islands'  : 'Kepulauan Bangka Belitung',
        'Riau Islands'             : 'Kepulauan Riau',
    }
    df_pop['provinsi'] = df_pop['provinsi'].replace(rename_prov)

    df_pop['population'] = (df_pop['population_thousands'] * 1000).astype(int)
    df_pop = df_pop.drop(columns=['population_thousands'])
    df_pop = df_pop.sort_values(['provinsi', 'tahun']).reset_index(drop=True)
 
    print(f"\nHasil: {len(df_pop)} records | {df_pop['provinsi'].nunique()} provinsi | tahun {sorted(df_pop['tahun'].unique())}")
    return df_pop
 

## Ekstrak Pengangguran dari BPS (CSV)

In [26]:
def build_unemployment_csv() -> pd.DataFrame:
    print("\n" + "=" * 60)
    print("EKSTRAKSI DATA PENGANGGURAN DARI CSV BPS")
    print("=" * 60)
 
    frames = []
    unemp_files = sorted(RAW_DIR.glob("Tingkat Pengangguran*.csv"))
 
    for filepath in unemp_files:
        year_match = re.search(r'(\d{4})\.csv$', filepath.name)
        if not year_match:
            continue
        year = int(year_match.group(1))
 
        try:
            df = pd.read_csv(filepath, skiprows=2, header=None, engine='python')
 
            if df.shape[1] < 2:
                print(f"  [SKIP] File kosong/tidak valid: {filepath.name}")
                continue
 
            col_names = ['provinsi', 'feb', 'agustus'] + [f'col_{i}' for i in range(df.shape[1] - 3)]
            df.columns = col_names[:df.shape[1]]
 
            df = df[['provinsi', 'feb', 'agustus']].copy()
            df = df.dropna(subset=['provinsi'])
 
            df = df[~df['provinsi'].astype(str).str.contains(
                'INDONESIA|Provinsi|Tingkat|Province|nan', case=False, na=True
            )]
 
            df['provinsi'] = df['provinsi'].astype(str).str.strip()
            df['feb']      = pd.to_numeric(df['feb'],     errors='coerce')
            df['agustus']  = pd.to_numeric(df['agustus'], errors='coerce')

            df['unemployment_rate_avg'] = df[['feb', 'agustus']].mean(axis=1)
            df = df.rename(columns={'feb': 'unemployment_rate_feb', 'agustus': 'unemployment_rate_aug'})
            df = df.dropna(subset=['unemployment_rate_avg'])
            df['tahun'] = year
            frames.append(df)
            print(f"  ✓ {filepath.name:<50} → {len(df)} provinsi")
 
        except Exception as e:
            print(f"  [ERROR] {filepath.name}: {e}")
 
    if not frames:
        print("  [WARN] Tidak ada data pengangguran yang berhasil diekstrak.")
        return pd.DataFrame(columns=['provinsi', 'tahun', 'unemployment_rate_feb',
                                     'unemployment_rate_aug', 'unemployment_rate_avg'])
 
    df_unemp = pd.concat(frames, ignore_index=True)
    df_unemp['provinsi'] = df_unemp['provinsi'].str.replace('Kep.', 'Kepulauan', regex=False)
    df_unemp['provinsi'] = df_unemp['provinsi'].str.replace('D.I. ', 'DI ', regex=False)
    df_unemp['provinsi'] = df_unemp['provinsi'].str.replace('D.K.I. ', 'DKI ', regex=False)
    rename_prov = {
        'Kep. Bangka Belitung'     : 'Kepulauan Bangka Belitung',
        'Kepulauan Bangka Belitung': 'Kepulauan Bangka Belitung',
        'Di Yogyakarta'            : 'DI Yogyakarta',
        'DI Yogyakarta'            : 'DI Yogyakarta',
        'DKI Jakarta'              : 'DKI Jakarta',
    }
    df_unemp['provinsi'] = df_unemp['provinsi'].replace(rename_prov)
    df_unemp = df_unemp.sort_values(['provinsi', 'tahun']).reset_index(drop=True)
 
    print(f"\nHasil: {len(df_unemp)} records | {df_unemp['provinsi'].nunique()} provinsi | tahun {sorted(df_unemp['tahun'].unique())}")
    return df_unemp

## Main

In [27]:
def main():
    print("\n" + "=" * 60)
    print("TAHAP 1 — EKSTRAKSI DATA MENTAH")
    print(f"Sumber  : {RAW_DIR}")
    print(f"Output  : {EXTRACTED_DIR}")
    print("=" * 60 + "\n")

    df_crime = build_crime_csv()
    out_crime = EXTRACTED_DIR / "crime_raw.csv"
    df_crime.to_csv(out_crime, index=False)
    print(f"\n✅ Disimpan: {out_crime}")
    print(df_crime.head(5).to_string(index=False))
 
    df_pop = build_population_csv()
    out_pop = EXTRACTED_DIR / "population_raw.csv"
    df_pop.to_csv(out_pop, index=False)
    print(f"\n✅ Disimpan: {out_pop}")
    print(df_pop.head(5).to_string(index=False))
 
    df_unemp = build_unemployment_csv()
    out_unemp = EXTRACTED_DIR / "unemployment_raw.csv"
    df_unemp.to_csv(out_unemp, index=False)
    print(f"\n✅ Disimpan: {out_unemp}")
    print(df_unemp.head(5).to_string(index=False))
 
    print("\n" + "=" * 60)
    print("SUMMARY EKSTRAKSI")
    print("=" * 60)
    print(f"crime_raw.csv       → {len(df_crime):>4} rows | kolom: {list(df_crime.columns)}")
    print(f"population_raw.csv  → {len(df_pop):>4} rows | kolom: {list(df_pop.columns)}")
    print(f"unemployment_raw.csv→ {len(df_unemp):>4} rows | kolom: {list(df_unemp.columns)}")
    print("\nTahap 1 selesai. Lanjut ke: src/02_cleaning.py")
 
 
if __name__ == "__main__":
    main()


TAHAP 1 — EKSTRAKSI DATA MENTAH
Sumber  : /Users/paulinadevinawijaya/Desktop/CRIME IN INDONESIA PORT/data/raw
Output  : /Users/paulinadevinawijaya/Desktop/CRIME IN INDONESIA PORT/data/extracted

EKSTRAKSI CRIME DATA DARI PDF BPS
  ✓ BPS-statistik-kriminal-2018.pdf          hal.99  tahun[2015, 2016, 2017]  → 32 provinsi
  ✓ BPS-statistik-kriminal-2021.pdf          hal.119  tahun[2018, 2019, 2020]  → 32 provinsi
  ✓ BPS-statistik-kriminal-2022.pdf          hal.111  tahun[2019, 2020, 2021]  → 33 provinsi
  ✓ BPS-statistik-kriminal-2023.pdf          hal.119  tahun[2020, 2021, 2022]  → 34 provinsi
  ✓ BPS-statistik-kriminal-2024.pdf          hal.123  tahun[2022, 2023]  → 33 provinsi
  ✓ PS-statistik-kriminal-2024-2025.pdf      hal.126  tahun[2023, 2024]  → 34 provinsi

Hasil: 331 records | 34 provinsi | tahun 2015–2024

✅ Disimpan: /Users/paulinadevinawijaya/Desktop/CRIME IN INDONESIA PORT/data/extracted/crime_raw.csv
provinsi  tahun  crime_total
    Aceh   2015         8048
    Aceh   201